# 🤖 2D Hilbert Curve Coverage Visualizer
### Hilbert-Augmented Reinforcement Learning for Multi-Robot Coverage

This notebook visualizes how a **Hilbert space-filling curve** guides robot coverage of a 2D grid,
compared to a naive **raster (boustrophedon) scan**.

**Concepts covered:**
- Hilbert curve generation at different orders (n = 1, 2, 3, 4)
- Raster scan baseline comparison
- Coverage efficiency and redundancy metrics
- Multi-robot partition via Hilbert index splitting
- Animated path visualization


In [ ]:
# ─────────────────────────────────────────
# CELL 1 — Install dependencies
# ─────────────────────────────────────────
!pip install hilbertcurve matplotlib numpy ipywidgets --quiet
print("✅ All dependencies installed!")

In [ ]:
# ─────────────────────────────────────────
# CELL 2 — Imports & Core Functions
# ─────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
from matplotlib.colors import LinearSegmentedColormap
from hilbertcurve.hilbertcurve import HilbertCurve
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed
import warnings
warnings.filterwarnings('ignore')

# ── Hilbert curve path generator ──────────────────────────────────────────────
def get_hilbert_path(order):
    """
    Returns an ordered list of (x, y) grid coordinates following the
    Hilbert space-filling curve of given order.
    Grid size = 2^order x 2^order.
    """
    n = 2 ** order          # grid side length
    total = n * n           # total cells
    hc = HilbertCurve(p=order, n=2)
    path = []
    for dist in range(total):
        coords = hc.point_from_distance(dist)
        path.append((coords[0], coords[1]))
    return path, n

# ── Raster (boustrophedon) scan generator ─────────────────────────────────────
def get_raster_path(order):
    """
    Returns an ordered list of (x, y) coordinates for a raster scan
    (row-by-row, alternating direction = boustrophedon pattern).
    """
    n = 2 ** order
    path = []
    for row in range(n):
        cols = range(n) if row % 2 == 0 else range(n - 1, -1, -1)
        for col in cols:
            path.append((col, row))
    return path, n

# ── Metric computation ────────────────────────────────────────────────────────
def compute_metrics(path, n):
    """
    Computes:
      - coverage_ratio   : fraction of unique cells visited
      - redundancy_rate  : fraction of revisited steps
      - total_distance   : sum of Euclidean step lengths
      - locality_score   : average step length (lower = more spatially local)
    """
    visited = set()
    revisits = 0
    total_dist = 0.0

    for i, cell in enumerate(path):
        if cell in visited:
            revisits += 1
        visited.add(cell)
        if i > 0:
            dx = path[i][0] - path[i-1][0]
            dy = path[i][1] - path[i-1][1]
            total_dist += np.sqrt(dx**2 + dy**2)

    total_cells = n * n
    coverage_ratio = len(visited) / total_cells
    redundancy_rate = revisits / len(path)
    locality_score = total_dist / len(path)
    return {
        'coverage_ratio': coverage_ratio,
        'redundancy_rate': redundancy_rate,
        'total_distance': total_dist,
        'locality_score': locality_score
    }

print("✅ Core functions loaded!")

In [ ]:
# ─────────────────────────────────────────
# CELL 3 — Static Side-by-Side Comparison
# Compare Hilbert vs Raster at different orders
# ─────────────────────────────────────────

def plot_comparison(order=3):
    """
    Plots Hilbert curve path vs Raster path side by side for a given order.
    Colormap shows traversal order (dark = start, bright = end).
    """
    hilbert_path, n = get_hilbert_path(order)
    raster_path, _ = get_raster_path(order)

    h_metrics = compute_metrics(hilbert_path, n)
    r_metrics = compute_metrics(raster_path, n)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.patch.set_facecolor('#0d1117')

    cmap_hilbert = LinearSegmentedColormap.from_list(
        'hilbert_cmap', ['#1a1a2e', '#16213e', '#0f3460', '#533483', '#e94560'], N=256)
    cmap_raster  = LinearSegmentedColormap.from_list(
        'raster_cmap',  ['#1a1a2e', '#1b4332', '#2d6a4f', '#52b788', '#d8f3dc'], N=256)

    paths = [hilbert_path, raster_path]
    cmaps = [cmap_hilbert, cmap_raster]
    titles = [f'🔴 Hilbert Curve  (order={order}, grid={n}×{n})',
              f'🟢 Raster Scan  (order={order}, grid={n}×{n})']
    metrics_list = [h_metrics, r_metrics]

    for ax, path, cmap, title, met in zip(axes, paths, cmaps, titles, metrics_list):
        ax.set_facecolor('#0d1117')
        total = len(path)

        # Draw grid background
        for x in range(n + 1):
            ax.axvline(x - 0.5, color='#ffffff0a', linewidth=0.3)
        for y in range(n + 1):
            ax.axhline(y - 0.5, color='#ffffff0a', linewidth=0.3)

        # Draw path segments colored by traversal order
        for i in range(1, total):
            t = i / total
            color = cmap(t)
            x_vals = [path[i-1][0], path[i][0]]
            y_vals = [path[i-1][1], path[i][1]]
            ax.plot(x_vals, y_vals, color=color, linewidth=1.2, alpha=0.85)

        # Mark start and end
        ax.scatter(*path[0],  color='#00ff88', s=80, zorder=5, label='Start')
        ax.scatter(*path[-1], color='#ff4444', s=80, zorder=5, label='End',
                   marker='X')

        ax.set_title(title, color='white', fontsize=13, pad=10, fontweight='bold')
        ax.set_xlim(-0.7, n - 0.3)
        ax.set_ylim(-0.7, n - 0.3)
        ax.set_aspect('equal')
        ax.tick_params(colors='#888888', labelsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor('#333333')

        # Metrics box
        metrics_text = (
            f"Coverage:    {met['coverage_ratio']*100:.1f}%\n"
            f"Redundancy:  {met['redundancy_rate']*100:.2f}%\n"
            f"Total dist:  {met['total_distance']:.1f} units\n"
            f"Locality:    {met['locality_score']:.3f} avg step"
        )
        ax.text(0.02, 0.02, metrics_text,
                transform=ax.transAxes,
                fontsize=9, color='#cccccc',
                verticalalignment='bottom',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='#1a1a2e',
                          edgecolor='#444', alpha=0.9),
                fontfamily='monospace')

        ax.legend(loc='upper right', fontsize=8,
                  facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')

    # Colorbar legend
    sm = plt.cm.ScalarMappable(cmap=cmap_hilbert)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, orientation='horizontal',
                        fraction=0.03, pad=0.08, aspect=40)
    cbar.set_label('Traversal order  (early → late)',
                   color='#aaaaaa', fontsize=10)
    cbar.ax.xaxis.set_tick_params(color='#aaaaaa')
    plt.setp(cbar.ax.xaxis.get_ticklabels(), color='#aaaaaa')

    plt.suptitle(
        f'Hilbert Curve vs Raster Scan — Grid Coverage (Order {order})',
        color='white', fontsize=15, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.savefig(f'coverage_comparison_order{order}.png',
                dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f"\n📊 Metrics Summary (Order {order}, {n}×{n} grid):")
    print(f"{'Metric':<22} {'Hilbert':>12} {'Raster':>12}")
    print("-" * 48)
    for key in h_metrics:
        hv = h_metrics[key]
        rv = r_metrics[key]
        unit = '%' if 'ratio' in key or 'rate' in key else ''
        scale = 100 if unit == '%' else 1
        print(f"{key:<22} {hv*scale:>11.3f}{unit} {rv*scale:>11.3f}{unit}")

# Run it!
plot_comparison(order=3)

In [ ]:
# ─────────────────────────────────────────
# CELL 4 — Interactive Widget (order slider)
# ─────────────────────────────────────────

order_slider = widgets.IntSlider(
    value=2, min=1, max=4, step=1,
    description='Curve Order:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

print("🎛️  Drag the slider to explore different Hilbert curve orders:")
print("   Order 1 → 2×2 grid  |  Order 2 → 4×4  |  Order 3 → 8×8  |  Order 4 → 16×16")
interact(plot_comparison, order=order_slider);

In [ ]:
# ─────────────────────────────────────────
# CELL 5 — Multi-Robot Hilbert Partitioning
# Split the Hilbert curve across N robots
# ─────────────────────────────────────────

def plot_multi_robot(order=3, num_robots=4):
    """
    Splits the Hilbert curve into equal partitions — one per robot.
    Each robot covers its own contiguous Hilbert index range.
    This mimics the paper's decentralized multi-robot strategy.
    """
    path, n = get_hilbert_path(order)
    total = len(path)
    chunk = total // num_robots

    robot_colors = [
        '#e94560', '#f5a623', '#50e3c2', '#4a90e2',
        '#bd10e0', '#7ed321', '#ff6b6b', '#ffd93d'
    ]

    fig, ax = plt.subplots(figsize=(9, 9))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')

    # Grid background
    for x in range(n + 1):
        ax.axvline(x - 0.5, color='#ffffff0a', linewidth=0.4)
    for y in range(n + 1):
        ax.axhline(y - 0.5, color='#ffffff0a', linewidth=0.4)

    legend_patches = []
    for r in range(num_robots):
        start_idx = r * chunk
        end_idx   = (r + 1) * chunk if r < num_robots - 1 else total
        segment   = path[start_idx:end_idx]
        color     = robot_colors[r % len(robot_colors)]

        # Shade background cells for this robot
        for (cx, cy) in segment:
            rect = mpatches.FancyBboxPatch(
                (cx - 0.45, cy - 0.45), 0.9, 0.9,
                boxstyle='round,pad=0.05',
                linewidth=0,
                facecolor=color, alpha=0.15
            )
            ax.add_patch(rect)

        # Draw path
        xs = [p[0] for p in segment]
        ys = [p[1] for p in segment]
        ax.plot(xs, ys, color=color, linewidth=1.8, alpha=0.9)

        # Robot start marker
        ax.scatter(xs[0], ys[0], color=color, s=100, zorder=6,
                   edgecolors='white', linewidths=0.8)
        ax.text(xs[0] + 0.15, ys[0] + 0.15, f'R{r+1}',
                color=color, fontsize=8, fontweight='bold', zorder=7)

        legend_patches.append(
            mpatches.Patch(color=color, label=f'Robot {r+1}  ({end_idx-start_idx} cells)')
        )

    ax.set_xlim(-0.7, n - 0.3)
    ax.set_ylim(-0.7, n - 0.3)
    ax.set_aspect('equal')
    ax.tick_params(colors='#666666')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')

    ax.legend(handles=legend_patches, loc='upper right', fontsize=9,
              facecolor='#1a1a2e', edgecolor='#444', labelcolor='white',
              framealpha=0.9)

    ax.set_title(
        f'Multi-Robot Hilbert Partitioning — {num_robots} Robots, '
        f'Order {order} ({n}×{n} grid)',
        color='white', fontsize=13, fontweight='bold', pad=12
    )

    # Overlap analysis
    all_visited = [set(path[r*chunk : (r+1)*chunk if r < num_robots-1 else total])
                   for r in range(num_robots)]
    overlap_pairs = 0
    for i in range(num_robots):
        for j in range(i+1, num_robots):
            overlap_pairs += len(all_visited[i] & all_visited[j])

    info = f"Total cells: {n*n}  |  Robots: {num_robots}  |  Overlap: {overlap_pairs} cells  |  Redundancy: {overlap_pairs/(n*n)*100:.2f}%"
    ax.text(0.5, -0.06, info, transform=ax.transAxes, ha='center',
            color='#aaaaaa', fontsize=9, fontfamily='monospace')

    plt.tight_layout()
    plt.savefig(f'multi_robot_order{order}_r{num_robots}.png',
                dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

# Run with interactive widgets
order_w  = widgets.IntSlider(value=3, min=2, max=4, step=1,
                              description='Order:', style={'description_width':'initial'})
robots_w = widgets.IntSlider(value=4, min=2, max=8, step=1,
                              description='Num Robots:', style={'description_width':'initial'})

print("🤖 Multi-Robot Coverage — Hilbert Curve Partitioning:")
interact(plot_multi_robot, order=order_w, num_robots=robots_w);

In [ ]:
# ─────────────────────────────────────────
# CELL 6 — Animated Coverage (Step by Step)
# Watch the robot trace the Hilbert curve live
# ─────────────────────────────────────────

def animate_coverage(order=2, mode='hilbert', speed=30):
    """
    Animates robot movement step by step along the selected path.
    Visited cells light up as coverage progresses.
    Displays coverage % counter live.
    """
    if mode == 'hilbert':
        path, n = get_hilbert_path(order)
        title_mode = 'Hilbert Curve'
        path_color = '#e94560'
        cell_color = '#533483'
    else:
        path, n = get_raster_path(order)
        title_mode = 'Raster Scan'
        path_color = '#52b788'
        cell_color = '#2d6a4f'

    total = len(path)
    visited_grid = np.zeros((n, n), dtype=float)

    fig, ax = plt.subplots(figsize=(7, 7))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')

    im = ax.imshow(
        visited_grid, origin='lower', cmap='plasma',
        vmin=0, vmax=1,
        extent=[-0.5, n - 0.5, -0.5, n - 0.5]
    )

    # Grid lines
    for x in range(n + 1):
        ax.axvline(x - 0.5, color='#ffffff15', linewidth=0.5)
    for y in range(n + 1):
        ax.axhline(y - 0.5, color='#ffffff15', linewidth=0.5)

    robot_dot, = ax.plot([], [], 'o', color='#ffffff', markersize=10,
                          markeredgecolor=path_color, markeredgewidth=2, zorder=6)
    trail_line, = ax.plot([], [], '-', color=path_color, linewidth=1.5,
                           alpha=0.6, zorder=5)

    title_obj = ax.set_title(
        f'{title_mode} — Order {order}  |  Coverage: 0.0%',
        color='white', fontsize=12, fontweight='bold'
    )

    ax.set_xlim(-0.7, n - 0.3)
    ax.set_ylim(-0.7, n - 0.3)
    ax.set_aspect('equal')
    ax.tick_params(colors='#555')
    for sp in ax.spines.values():
        sp.set_edgecolor('#333')

    visited_set = set()
    trail_x, trail_y = [], []

    def update(frame):
        cx, cy = path[frame]
        visited_grid[cy, cx] = 0.6 + 0.4 * (frame / total)
        im.set_data(visited_grid.copy())

        visited_set.add((cx, cy))
        trail_x.append(cx)
        trail_y.append(cy)

        # Keep trail short (last 20 steps)
        if len(trail_x) > 20:
            trail_x.pop(0)
            trail_y.pop(0)

        robot_dot.set_data([cx], [cy])
        trail_line.set_data(trail_x, trail_y)

        coverage_pct = len(visited_set) / (n * n) * 100
        title_obj.set_text(
            f'{title_mode} — Order {order}  |  '
            f'Step {frame+1}/{total}  |  Coverage: {coverage_pct:.1f}%'
        )
        return im, robot_dot, trail_line, title_obj

    interval = max(10, 200 - speed * 3)  # ms between frames
    ani = animation.FuncAnimation(
        fig, update, frames=total,
        interval=interval, blit=True, repeat=False
    )

    plt.tight_layout()
    # Render inline
    display(HTML(ani.to_jshtml()))
    plt.close()

# Interactive controls
order_w2 = widgets.IntSlider(value=2, min=1, max=3, step=1,
                              description='Order:', style={'description_width':'initial'})
mode_w   = widgets.Dropdown(options=['hilbert', 'raster'], value='hilbert',
                             description='Mode:', style={'description_width':'initial'})
speed_w  = widgets.IntSlider(value=50, min=10, max=90, step=10,
                              description='Speed:', style={'description_width':'initial'})

print("🎬 Animated Coverage — watch the robot trace the path:")
print("  ⚠️  Order 3 has 64 steps, Order 4 has 256 — start with Order 2 for speed")
interact(animate_coverage, order=order_w2, mode=mode_w, speed=speed_w);

In [ ]:
# ─────────────────────────────────────────
# CELL 7 — Redundancy Heatmap Analyzer
# Simulate noisy robot paths and measure revisits
# ─────────────────────────────────────────

def redundancy_heatmap(order=3, noise_prob=0.1, mode='hilbert'):
    """
    Simulates a robot following a Hilbert or raster path but with random
    positional noise (revisiting nearby cells), and produces a redundancy
    heatmap showing how many times each cell was visited.
    """
    if mode == 'hilbert':
        path, n = get_hilbert_path(order)
        cmap_name = 'Reds'
        label_color = '#e94560'
    else:
        path, n = get_raster_path(order)
        cmap_name = 'Greens'
        label_color = '#52b788'

    visit_count = np.zeros((n, n), dtype=int)

    np.random.seed(42)
    for (cx, cy) in path:
        # Noisy visit: with some probability, visit a neighboring cell too
        visit_count[cy, cx] += 1
        if np.random.random() < noise_prob:
            # Random neighbor
            dx, dy = np.random.choice([-1, 0, 1], size=2)
            nx_, ny_ = cx + dx, cy + dy
            if 0 <= nx_ < n and 0 <= ny_ < n:
                visit_count[ny_, nx_] += 1

    total_visits  = visit_count.sum()
    covered_cells = (visit_count > 0).sum()
    redundant     = visit_count[visit_count > 1].sum() - (visit_count > 1).sum()
    coverage_pct  = covered_cells / (n * n) * 100
    redundancy_pct = redundant / total_visits * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor('#0d1117')

    # Left: heatmap
    ax1 = axes[0]
    ax1.set_facecolor('#0d1117')
    im = ax1.imshow(visit_count, cmap=cmap_name, origin='lower',
                    extent=[-0.5, n-0.5, -0.5, n-0.5])
    cbar = plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
    cbar.set_label('Visit count', color='#aaa', fontsize=9)
    cbar.ax.yaxis.set_tick_params(color='#aaa')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#aaa')

    ax1.set_title(f'Redundancy Heatmap — {mode.capitalize()} (noise={noise_prob})',
                  color='white', fontsize=11, fontweight='bold')
    ax1.tick_params(colors='#555')
    for sp in ax1.spines.values():
        sp.set_edgecolor('#333')

    # Right: bar chart of metrics
    ax2 = axes[1]
    ax2.set_facecolor('#0d1117')
    bars = ax2.bar(
        ['Coverage\n(%)', 'Redundancy\n(%)', 'Noise\nProb (%)'],
        [coverage_pct, redundancy_pct, noise_prob * 100],
        color=[label_color, '#ff6b6b', '#f5a623'],
        edgecolor='#222', linewidth=0.8, width=0.5
    )
    for bar, val in zip(bars, [coverage_pct, redundancy_pct, noise_prob*100]):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom',
                 color='white', fontsize=10, fontweight='bold')

    ax2.set_ylim(0, 115)
    ax2.set_title('Coverage & Redundancy Metrics',
                  color='white', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Percentage (%)', color='#aaa', fontsize=9)
    ax2.tick_params(colors='#888')
    ax2.set_facecolor('#0d1117')
    for sp in ax2.spines.values():
        sp.set_edgecolor('#333')

    info = (f"Grid: {n}×{n} | Total visits: {total_visits} | "
            f"Covered: {covered_cells}/{n*n} cells")
    fig.text(0.5, 0.01, info, ha='center', color='#666', fontsize=8,
             fontfamily='monospace')

    plt.tight_layout()
    plt.savefig(f'heatmap_{mode}_order{order}_noise{int(noise_prob*100)}.png',
                dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

# Interactive controls
order_w3 = widgets.IntSlider(value=3, min=2, max=4, step=1,
                              description='Order:', style={'description_width':'initial'})
noise_w  = widgets.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.05,
                                description='Noise prob:', style={'description_width':'initial'},
                                readout_format='.2f')
mode_w2  = widgets.Dropdown(options=['hilbert', 'raster'], value='hilbert',
                             description='Mode:', style={'description_width':'initial'})

print("🔥 Redundancy Heatmap — simulates noisy robot paths:")
interact(redundancy_heatmap, order=order_w3, noise_prob=noise_w, mode=mode_w2);

In [ ]:
# ─────────────────────────────────────────
# CELL 8 — Locality Preservation Analysis
# Shows WHY Hilbert curves are better than raster
# ─────────────────────────────────────────

def locality_analysis():
    """
    For each order, computes the average physical jump distance
    between consecutive steps for both Hilbert and Raster paths.
    Lower = better spatial locality.
    """
    orders = [1, 2, 3, 4]
    hilbert_locality = []
    raster_locality  = []
    hilbert_maxjump  = []
    raster_maxjump   = []

    for order in orders:
        hp, n = get_hilbert_path(order)
        rp, _ = get_raster_path(order)

        h_dists = [np.sqrt((hp[i][0]-hp[i-1][0])**2 + (hp[i][1]-hp[i-1][1])**2)
                   for i in range(1, len(hp))]
        r_dists = [np.sqrt((rp[i][0]-rp[i-1][0])**2 + (rp[i][1]-rp[i-1][1])**2)
                   for i in range(1, len(rp))]

        hilbert_locality.append(np.mean(h_dists))
        raster_locality.append(np.mean(r_dists))
        hilbert_maxjump.append(np.max(h_dists))
        raster_maxjump.append(np.max(r_dists))

    x = np.arange(len(orders))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0d1117')

    for ax in axes:
        ax.set_facecolor('#111827')
        for sp in ax.spines.values():
            sp.set_edgecolor('#333')
        ax.tick_params(colors='#888')
        ax.yaxis.label.set_color('#aaa')
        ax.xaxis.label.set_color('#aaa')

    # Avg step distance
    b1 = axes[0].bar(x - width/2, hilbert_locality, width,
                     label='Hilbert', color='#e94560', alpha=0.85)
    b2 = axes[0].bar(x + width/2, raster_locality, width,
                     label='Raster', color='#52b788', alpha=0.85)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f'Order {o}\n({2**o}×{2**o})' for o in orders])
    axes[0].set_ylabel('Avg step distance (grid units)')
    axes[0].set_title('Average Step Distance (Locality)', color='white',
                      fontsize=12, fontweight='bold')
    axes[0].legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')
    axes[0].set_facecolor('#111827')

    for bar in b1:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{bar.get_height():.2f}', ha='center', va='bottom',
                     color='white', fontsize=8)
    for bar in b2:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{bar.get_height():.2f}', ha='center', va='bottom',
                     color='white', fontsize=8)

    # Max jump distance
    axes[1].plot(orders, hilbert_maxjump, 'o-', color='#e94560',
                 linewidth=2, markersize=8, label='Hilbert max jump')
    axes[1].plot(orders, raster_maxjump, 's-', color='#52b788',
                 linewidth=2, markersize=8, label='Raster max jump')
    axes[1].fill_between(orders, hilbert_maxjump, raster_maxjump,
                          alpha=0.1, color='#ffffff')
    axes[1].set_xticks(orders)
    axes[1].set_xticklabels([f'Order {o}' for o in orders])
    axes[1].set_ylabel('Max jump distance (grid units)')
    axes[1].set_title('Worst-case Jump (Locality Failure)', color='white',
                       fontsize=12, fontweight='bold')
    axes[1].legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')
    axes[1].set_facecolor('#111827')

    fig.suptitle(
        'Spatial Locality Comparison — Hilbert vs Raster\n'
        '(Lower avg step = better locality = less robot energy wasted)',
        color='white', fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('locality_analysis.png', dpi=150,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show()

    print("\n📊 Locality Summary:")
    print(f"{'Order':<8} {'H avg':>10} {'R avg':>10} {'H max':>10} {'R max':>10}")
    print("-" * 50)
    for i, o in enumerate(orders):
        print(f"  {o:<6} {hilbert_locality[i]:>10.3f} {raster_locality[i]:>10.3f} "
              f"{hilbert_maxjump[i]:>10.3f} {raster_maxjump[i]:>10.3f}")

locality_analysis()

## 📚 Summary

| Cell | What it shows | Paper connection |
|------|--------------|------------------|
| 3 | Static Hilbert vs Raster comparison | Section: Hilbert-based spatial indices |
| 4 | Interactive order slider | Scalability experiments |
| 5 | Multi-robot Hilbert partitioning | Decentralized multi-robot execution |
| 6 | Animated robot coverage | Waypoint interface & trajectory following |
| 7 | Redundancy heatmap with noise | Coverage efficiency & redundancy metrics |
| 8 | Locality preservation analysis | Why geometric priors help RL |

### Key insight
The Hilbert curve's spatial locality property means nearby points on the curve are also physically nearby — reducing the energy a robot wastes on long jumps and making it a natural prior for structured exploration in RL agents.
